# Fase 1 — holonomía en el grafo de pares cripto

Resultados sobre una semana de cotizaciones de Alpaca (10 a 16 de noviembre de 2026),
26 pares y 9 monedas. El cuaderno **no calcula nada**: llama al paquete y grafica.
Las tablas vienen de `scripts/medir.py`; para regenerarlas, correr ese script.

Antes de mirar un solo número: `python -m pytest` tiene que pasar entero.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import figuras
from holonomia import grafo as G
from capturar import SUBGRAFO

RES = pathlib.Path.cwd().parent / 'resultados'
DIA = '2026-11-10'

## 1. El complejo

Lo primero no es un número del mercado sino del grafo: si los triángulos rellenan
todo el espacio de ciclos, **la parte armónica es cero por construcción** y medirla
no dice nada del mercado.

In [ ]:
g = G.desde_simbolos(SUBGRAFO)
print(f'monedas        {g.n_monedas}')
print(f'pares          {g.n_aristas}')
print(f'triángulos     {g.n_caras}')
print(f'rango ciclos   {g.rango_ciclos()}')
print(f'b1             {g.betti1()}')

completo = G.desde_simbolos([l.strip() for l in open(RES / 'pares_alpaca.txt')]) \
    if (RES / 'pares_alpaca.txt').exists() else None
if completo is not None:
    print()
    print(f'grafo completo de Alpaca: V={completo.n_monedas} E={completo.n_aristas} '
          f'triángulos={completo.n_caras} b1={completo.betti1()}')

## 2. Qué se descartó

Regla 3 del proyecto: cada resultado empírico lleva su registro de descartes.

In [ ]:
descartes = pd.read_csv(RES / 'descartes.csv')
display(descartes)

## 3. Distribución de la holonomía frente al medio spread

La pregunta de §2.5: ¿cuánta curvatura hay, y cómo se compara con lo que cuesta
cruzar el ciclo?

In [ ]:
serie = figuras.serie_insignia(DIA)
print(f'{len(serie):,} instantes sincronizados a Δt = 1 s')
display(serie[['h_mid', 'h_exec', 'medio_spread', 'umbral']].describe().T * 1e4)
figuras.figura_distribucion(serie, DIA)

In [ ]:
resumen = pd.read_csv(RES / 'resumen_por_triangulo.csv')
por_ciclo = resumen[resumen.instantes > 1000].groupby('ciclo').median(numeric_only=True)
display(por_ciclo[['instantes', 'mediana_abs_h_mid_pb', 'p99_abs_h_mid_pb',
                   'mediana_medio_spread_pb', 'mediana_umbral_pb',
                   'frac_fuera_del_spread', 'frac_h_exec_positiva']]
        .sort_values('mediana_abs_h_mid_pb', ascending=False))
figuras.figura_curvatura_contra_costo(resumen)

## 4. ¿Es asincronía?

Dos vistas del mismo control. La primera cambia la ventana Δt; la segunda mira la
antigüedad **dentro** de una ventana fija, que es la que no confunde rancidez con
selección de ráfagas de actividad.

In [ ]:
figuras.figura_sensibilidad(pd.read_csv(RES / 'sensibilidad_delta_t.csv'))

In [ ]:
from holonomia import datos, limpieza, medidas
df = datos.cargar_quotes(pathlib.Path.cwd().parent / 'datos_crudos' / f'quotes-{DIA}')
limpio, _ = limpieza.filtrar(df)
p = limpieza.panel(limpio, figuras.PATAS)
display(medidas.holonomia_por_antiguedad(p, figuras.INSIGNIA, delta_t=1.0))

## 5. Persistencia

¿Las inconsistencias se cierran en milisegundos, segundos o minutos?

In [ ]:
figuras.figura_persistencia(pd.read_csv(RES / 'persistencia.csv'))

## 6. Reparto rotacional contra armónico

El complejo vivo cambia instante a instante: una arista que no cotizó dentro de Δt
sale del complejo, y con ella pueden irse sus caras. Ahí es donde puede aparecer
parte armónica en un grafo que, completo, no la admite.

In [ ]:
figuras.figura_reparto(pd.read_csv(RES / 'reparto_hodge.csv'))

## 7. Régimen

¿Hay más curvatura en los momentos de estrés?

In [ ]:
figuras.figura_regimen(serie, pd.read_csv(RES / 'actividad.csv'), DIA)